## 👀 Vistas (Views) en Databricks

A medida que los proyectos de Data Engineering crecen, las consultas SQL también aumentan en complejidad. Es común encontrar consultas con múltiples:

* 🔗 JOIN
* 📊 Agregaciones
* 🔄 Subconsultas
* 🪟 Funciones de ventana
* 📑 Expresiones condicionales

Reutilizar este tipo de consultas una y otra vez puede dificultar el mantenimiento y la comprensión del código. Sin embargo, para resolver este problema, Databricks incorpora las **Views (Vistas)**.

Una vista no almacena datos físicamente; simplemente encapsula una consulta SQL para reutilizarla posteriormente como si fuera una tabla.

---

### 🚀 ¿Por qué utilizar una Vista?

Las vistas ofrecen múltiples ventajas dentro de una arquitectura de datos:

* ✅ Simplifican consultas complejas.
* ✅ Favorecen la reutilización de lógica SQL.
* ✅ Facilitan el mantenimiento del código.
* ✅ Permiten abstraer la estructura física de las tablas.
* ✅ Expone únicamente información necesaria, ocultando columnas sensibles o información PII cuando sea requerido.

> En lugar de ejecutar una consulta de decenas de líneas, basta con consultar la vista.

---

### 🗂️ Tipos de Vistas en Databricks

Databricks ofrece tres tipos principales de vistas, cada una diseñada para un escenario diferente:

* 📚 Vista Persistente (Persistent View)
* ⏳ Vista Temporal (Temporary View)
* 🌐 Vista Temporal Global (Global Temporary View)

---

### 📊 Comparación

| Característica                          | Persistent View    | Temporary View | Global Temporary View                              |
| --------------------------------------- | ------------------ | -------------- | -------------------------------------------------- |
| Forma parte de Unity Catalog            | ✅                  | ❌              | ❌                                                  |
| Persiste después de finalizar la sesión | ✅                  | ❌              | ❌                                                  |
| Disponible para otros usuarios          | ✅ (según permisos) | ❌              | ✅ (mientras la aplicación Spark permanezca activa) |
| Almacena datos físicamente              | ❌                  | ❌              | ❌                                                  |

---

### 🎓 Conclusión

Las tres variantes comparten el mismo objetivo: **abstraer y reutilizar consultas SQL**.

La diferencia principal radica en **su alcance y persistencia**:

* 📚 **Persistent View:** cuando la consulta debe formar parte del modelo de datos y mantenerse disponible en Unity Catalog.

* ⏳ **Temporary View:** cuando la consulta solo será utilizada durante la sesión actual.

* 🌐 **Global Temporary View:** cuando la consulta debe compartirse temporalmente entre distintos notebooks o sesiones de trabajo, sin convertirse en un objeto permanente del catálogo.


### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("11Views").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

#### 🔍 ¿Qué acaba de ocurrir?

Acabamos de crear una Spark Session. Esta sesión representa nuestro punto de entrada hacia:
* 🏗️ Apache Spark
* 🏗️ Databricks Runtime
* 🏗️ Delta Lake
* 🏗️ Unity Catalog

### 📚 Vista Persistente (Persistent View)

Es la vista tradicional que encontramos en la mayoría de motores de bases de datos. Forma parte de la gobernanza de **Unity Catalog**, por lo que permanece disponible hasta que sea eliminada explícitamente.

* Sintaxis
    ```sql id="0sjgnj"
    CREATE OR REPLACE VIEW catalog.schema.nombre_view
    AS
    SELECT ...
    FROM ...;
    ```

* ¿Cuándo utilizarla?

    * ✅ Consultas reutilizadas frecuentemente.
    * ✅ Modelos analíticos.
    * ✅ Exposición de información para otros usuarios.
    * ✅ Objetos permanentes dentro del Lakehouse.


In [0]:
### VISTAS PERSISTENTES (PERSISTENT VIEW)

#### PASO 1. PREPARAR QUERY SQL
display(spark.sql("""
                  
                  SELECT initcap(country) as Country,customer_id as CustomerID,
                  amount as TotalSales,payment_method as PaymentMethod 
                  FROM catalog_databricks_2026_de.schema_databricks_2026_de.delta_table_stream_kinesis
                  WHERE payment_method REGEXP '^[A-Z]$'; 
                  """))

#### PASO 2. ACOPLAR QUERY A UNA VISTA PERSISTENTE
spark.sql("""
          
        CREATE OR REPLACE VIEW catalog_databricks_2026_de.schema_databricks_2026_de.persistent_view
        AS
        SELECT initcap(country) as Country,customer_id as CustomerID,
        amount as TotalSales,payment_method as PaymentMethod 
        FROM catalog_databricks_2026_de.schema_databricks_2026_de.delta_table_stream_kinesis
        WHERE payment_method REGEXP '^[A-Z]$'; 
          
          """)
print("Vista Persistente definida correctamente")

#### PASO 3. VERIFICAR VISTA

display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.persistent_view")) ## DATOS DE LA VISTA PERSISTENTE

display(spark.sql("DESCRIBE TABLE EXTENDED catalog_databricks_2026_de.schema_databricks_2026_de.persistent_view")) ## ESTRUCTURA DE LA VISTA PERSISTENTE

"""

✔️ Resultado: Type - VIEW

💡En Unity Catalog, las vistas persistentes se muestran junto con las tablas dentro del mismo 
  esquema en el Catalog Explorer. Esto no significa que una vista se haya convertido en una
  tabla; simplemente Databricks unificó la representación visual de todos los objetos gobernados
  por Unity Catalog.

"""
print()

### ⏳ Vista Temporal (Temporary View)

Las vistas temporales existen únicamente durante la sesión actual de Spark.

No forman parte de Unity Catalog y desaparecen automáticamente cuando finaliza la sesión o el clúster deja de ejecutarse.

* Sintaxis
    ```sql id="mbf0hk"
    CREATE OR REPLACE TEMPORARY VIEW nombre_view
    AS
    SELECT ...
    FROM ...;
    ```

* ¿Cuándo utilizarla?

    * ✅ Procesamientos intermedios.
    * ✅ Transformaciones temporales.
    * ✅ Pruebas.
    * ✅ Desarrollo interactivo dentro de notebooks.

In [0]:
### VISTAS TEMPORALES (TEMPORARY VIEW)

#### PASO 1. PREPARAR QUERY SQL
display(spark.sql("""
                  
                  SELECT initcap(country) as Country,customer_id as CustomerID,
                  amount as TotalSales,payment_method as PaymentMethod 
                  FROM catalog_databricks_2026_de.schema_databricks_2026_de.delta_table_stream_kinesis
                  WHERE payment_method REGEXP '^[A-Z]$'; 
                  """))

#### PASO 2. ACOPLAR QUERY A UNA VISTA TEMPORAL
spark.sql("""
          
        CREATE OR REPLACE TEMPORARY VIEW temporary_view
        AS
        SELECT initcap(country) as Country,customer_id as CustomerID,
        amount as TotalSales,payment_method as PaymentMethod 
        FROM catalog_databricks_2026_de.schema_databricks_2026_de.delta_table_stream_kinesis
        WHERE payment_method REGEXP '^[A-Z]$'; 
          
          """)
print("Vista Temporal definida correctamente")

#### PASO 3. VERIFICAR VISTA TEMPORAL

display(spark.sql("SELECT * FROM temporary_view")) ## DATOS DE LA VISTA TEMPORAL

display(spark.sql("DESCRIBE TABLE EXTENDED temporary_view")) ## ESTRUCTURA DE LA VISTA TEMPORAL

"""

   💡 La vista temporal al no estar amarrada a un catalago y esquema específico, no muestra esos
      metadatos. Asimismo, al cerrar y abrir Databricks ya no existira la vista.

"""

### 🌐 Vista Temporal Global (Global Temporary View)

Las Global Temporary Views también son temporales, pero presentan una diferencia importante.

Aunque no pertenecen a Unity Catalog, pueden ser compartidas entre diferentes notebooks o sesiones dentro del mismo Workspace, siempre que continúe activa la aplicación Spark.

### Sintaxis

```sql id="pktyor"
CREATE OR REPLACE GLOBAL TEMPORARY VIEW nombre_view
AS
SELECT ...
FROM ...;
```

### ¿Cuándo utilizarla?

* ✅ Compartir resultados temporales entre notebooks.
* ✅ Procesos colaborativos durante una misma ejecución.
* ✅ Escenarios donde la vista no necesita persistir de forma permanente.

In [0]:
### VISTAS TEMPORALES GLOBALES (GLOBAL TEMPORARY VIEW)

#### PASO 1. PREPARAR QUERY SQL
# display(spark.sql("""
                  
#                   SELECT initcap(country) as Country,customer_id as CustomerID,
#                   amount as TotalSales,payment_method as PaymentMethod 
#                   FROM catalog_databricks_2026_de.schema_databricks_2026_de.delta_table_stream_kinesis
#                   WHERE payment_method REGEXP '^[A-Z]$'; 
#                   """))

#### PASO 2. ACOPLAR QUERY A UNA VISTA TEMPORAL GLOBAL
spark.sql("""
          
        CREATE OR REPLACE GLOBAL TEMPORARY VIEW global_temporary_view
        AS
        SELECT initcap(country) as Country,customer_id as CustomerID,
        amount as TotalSales,payment_method as PaymentMethod 
        FROM catalog_databricks_2026_de.schema_databricks_2026_de.delta_table_stream_kinesis
        WHERE payment_method REGEXP '^[A-Z]$'; 
          
          """)
print("Vista Global Temporal definida correctamente")

#### PASO 3. VERIFICAR VISTA TEMPORAL

display(spark.sql("SELECT * FROM global_temporary_view")) ## DATOS DE LA VISTA GLOBAL TEMPORAL

display(spark.sql("DESCRIBE TABLE EXTENDED global_temporary_view")) ## ESTRUCTURA DE LA VISTA GLOBAL TEMPORAL

"""

   💡 La vista global temporal al no estar amarrada a un catalago y esquema específico, no muestra esos
      metadatos. Asimismo, al cverrar y abrir Databricks ya no existira la vista.
   
   💡 En este caso, las vistas temporales globales no pueden definirse en Databricks Free Edition. Eso se debe
      a que no existe compartición de recursos y solo utilizamos un compute serverless. 

"""